In [ ]:
from pathlib import Path

# General helpful packages for data analysis and visualization
import pandas as pd
import scanpy as sc
import seaborn as sns
from muon import atac as ac  # the module containing function for scATAC data processing
import matplotlib.pyplot as plt
import muon as mu

import sys
PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
SRC_DIR = str(PROJECT_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
    
import multiomic_transformer.utils.muon_preprocessing as muon_prep

In [ ]:
RAW_DATA_DIR=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/RAW_DATA")

cell_type="Macrophage"
sample_name="buffer_1"
experiment_name=f"{cell_type}_{sample_name}_tutorial"
organism_code="hg38"

tss_path=f"{PROJECT_DIR}/data/genome_data/genome_annotation/{organism_code}/gene_tss.bed"
tf_list_path=""

RAW_DATASET_NAME="Macrophage_10x_raw"
RAW_DATA_DIR=f"{PROJECT_DIR}/data/raw/{RAW_DATASET_NAME}/"
PROCESSED_DATA_DIR=f"{PROJECT_DIR}/data/processed/{experiment_name}"

FRAG_PATH=f"{RAW_DATA_DIR}/{sample_name}/fragments.sorted.tsv.gz"

RNA_COUNT_FILE=""
ATAC_COUNT_FILE=""

RAW_H5_FILE=""

SAMPLE_DATA_DIR = Path(RAW_DATA_DIR) / sample_name
SAMPLE_PROCESSED_DATA_DIR = Path(PROCESSED_DATA_DIR) / sample_name

In [ ]:
tss_path = Path(tss_path)
rna_count_file = Path(RNA_COUNT_FILE) if RNA_COUNT_FILE else None
atac_count_file = Path(ATAC_COUNT_FILE) if ATAC_COUNT_FILE else None
raw_h5_file = Path(RAW_H5_FILE) if RAW_H5_FILE else None
tf_list_file = Path(tf_list_path) if tf_list_path else None
frag_path = Path(FRAG_PATH) if FRAG_PATH else None

In [ ]:
filtering_setting_df = pd.read_csv(PROJECT_DIR / "dev" / "notebooks" / "muon_preprocessing" /"qc_filtering_settings.tsv", sep="\t")
sample_filtering_settings = filtering_setting_df[filtering_setting_df["Sample"] == sample_name]    

# ----- RNA QC thresholds -----
MIN_CELLS_PER_GENE = muon_prep.get_threshold(sample_filtering_settings, "Min Cells per Gene")
MIN_GENES_PER_CELL = muon_prep.get_threshold(sample_filtering_settings, "Min Genes per Cell")
MAX_GENES_PER_CELL = muon_prep.get_threshold(sample_filtering_settings, "Max Genes per Cell")
MIN_TOTAL_COUNTS = muon_prep.get_threshold(sample_filtering_settings, "Min Total Counts")
MAX_TOTAL_COUNTS = muon_prep.get_threshold(sample_filtering_settings, "Max Total Counts")
MAX_PCT_COUNTS_MT = muon_prep.get_threshold(sample_filtering_settings, "Max Pct MT")

# ----- ATAC QC thresholds -----
MIN_CELLS_PER_PEAK = muon_prep.get_threshold(sample_filtering_settings, "Min Cells per Peak")
MIN_PEAKS_PER_CELL = muon_prep.get_threshold(sample_filtering_settings, "Min Peaks per Cell")
MAX_PEAKS_PER_CELL = muon_prep.get_threshold(sample_filtering_settings, "Max Peaks per Cell")
MIN_TOTAL_PEAK_COUNTS = muon_prep.get_threshold(sample_filtering_settings, "Min Total Peak Counts")
MAX_TOTAL_PEAK_COUNTS = muon_prep.get_threshold(sample_filtering_settings, "Max Total Peak Counts")

if not SAMPLE_PROCESSED_DATA_DIR.exists():
    SAMPLE_PROCESSED_DATA_DIR.mkdir(parents=True)

In [ ]:
mdata, frag_path = muon_prep.load_raw_data(sample_name, SAMPLE_DATA_DIR, rna_count_file, atac_count_file, raw_h5_file)

mdata.write(SAMPLE_PROCESSED_DATA_DIR / f"{sample_name}.h5mu")

In [ ]:
data_processor = muon_prep.MudataProcessor(
    mdata=mdata,
    processed_data_dir=SAMPLE_PROCESSED_DATA_DIR,
    sample_name=sample_name,
    tss_path=tss_path,
    tf_list_file=tf_list_file
)

In [ ]:
# RNA QC and Preprocessing
data_processor.rna_qc_filter(
    min_cells_per_gene = MIN_CELLS_PER_GENE,
    min_genes_per_cell = MIN_GENES_PER_CELL,
    max_genes_per_cell = MAX_GENES_PER_CELL,
    min_total_counts_per_cell = MIN_TOTAL_COUNTS,
    max_total_counts_per_cell = MAX_TOTAL_COUNTS,
    max_pct_counts_mt = MAX_PCT_COUNTS_MT,
    norm_target_sum = 1e4,
    min_rna_disp = 0.5,
    filter_hvgs = False,
    tf_list_file = None,
    fig_dir=SAMPLE_PROCESSED_DATA_DIR / "preprocessing_figures" / "rna_qc",
    )

In [ ]:
data_processor.rna_pca_and_neighbors(
    data_processor.rna, 
    n_pcs=20,
    n_neighbors=10,
    fig_dir=SAMPLE_PROCESSED_DATA_DIR / "preprocessing_figures" / "rna_qc",
    )

In [ ]:
# ATAC QC and Preprocessing
data_processor.atac_qc_filter(
    min_cells_per_peak=MIN_CELLS_PER_PEAK,
    min_peaks_per_cell=MIN_PEAKS_PER_CELL,
    max_peaks_per_cell=MAX_PEAKS_PER_CELL,
    min_total_counts_per_cell=MIN_TOTAL_PEAK_COUNTS,
    max_total_counts_per_cell=MAX_TOTAL_PEAK_COUNTS,
    min_atac_disp=0.5,
    promoter_upstream=1000,
    promoter_downstream=100,
    distal_max=200_000,
    filter_hvgs=False,
    fig_dir=SAMPLE_PROCESSED_DATA_DIR / "preprocessing_figures" / "atac_qc",
    )

In [ ]:
data_processor.nucleosome_signal(
    frag_path=frag_path, 
    fig_dir=SAMPLE_PROCESSED_DATA_DIR / "preprocessing_figures" / "atac_qc"
    )

In [ ]:
data_processor.tss_enrichment(
    frag_path=frag_path, 
    n_tss=500, 
    extend_upstream=1000, 
    extend_downstream=1000,
    fig_dir=SAMPLE_PROCESSED_DATA_DIR / "preprocessing_figures" / "atac_qc"
    )


In [ ]:
# Save the processed data
muon_prep.save_processed_data(data_processor.mdata, SAMPLE_PROCESSED_DATA_DIR)

In [ ]:
# Integrate the RNA and ATAC modalities using MOFA+
muon_prep.integrate_rna_atac(
    data_processor.mdata, 
    SAMPLE_PROCESSED_DATA_DIR, 
    sample_name, 
    fig_dir=SAMPLE_PROCESSED_DATA_DIR / "integration"
    )

In [ ]:
# Create metacells
muon_prep.create_metacells(data_processor.mdata, SAMPLE_PROCESSED_DATA_DIR, hops=2)